In [1]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║         🧠  N O T E G E N I U S  —  Mistral-7B Fine-Tune Pipeline         ║
# ║                                                                              ║
# ║  Model   : Mistral-7B-Instruct-v0.2  (7B params)                           ║
# ║  Method  : QLoRA (4-bit NF4) via PEFT — fits on Kaggle T4 x2              ║
# ║  Data    : Raw text extracted from your class-notes PDFs (unsupervised)     ║
# ║  Deploy  : Gradio RAG chat UI with share=True (public URL)                 ║
# ║  Hardware: Kaggle T4 x2  →  Settings › Accelerator  (required)            ║
# ╚══════════════════════════════════════════════════════════════════════════════╝
#
#  PIPELINE
#  ────────
#  Step 1  Install requirements
#  Step 2  Load training data      (PDFs → raw text corpus)
#  Step 3  Preprocess data         (tokenise into fixed-length windows)
#  Step 4  Set up training args    (epochs, LR, batch size, logging)
#  Step 5  Initialise model        (Mistral-7B 4-bit QLoRA + LoRA adapter)
#  Step 6  Train the model         (Trainer causal-LM loop)
#  Step 7  Evaluate per epoch      (loss curve + perplexity table)
#  Step 8  Save fine-tuned model   (adapter weights + merged full model)
#  Step 9  Deploy                  (Gradio RAG chat with the fine-tuned model)
#
#  QUICK START
#  ───────────
#  1. Add your PDF dataset to Kaggle  (Settings › Add Data)
#  2. Enable GPU T4 x2               (Settings › Accelerator)
#  3. Run all cells top-to-bottom — a public Gradio URL prints at the end
 
 
# ══════════════════════════════════════════════════════════════════════════════
# STEP 1 · Install Requirements
# ══════════════════════════════════════════════════════════════════════════════
 
import os, subprocess, sys
 
os.environ["PIP_ROOT_USER_ACTION"] = "ignore"
 
# Dependency notes
# ─ peft         : LoRA adapter injection & merging
# ─ bitsandbytes : 4-bit NF4 quantisation (QLoRA) — Mistral-7B needs this on T4
# ─ accelerate   : multi-GPU / mixed-precision training support
# ─ transformers : Mistral model, tokeniser, Trainer
# ─ datasets     : HuggingFace Dataset used by Trainer
# ─ pypdf        : PDF text extraction to build the raw text corpus
# ─ scikit-learn : TF-IDF retriever for the RAG pipeline in Step 9
# ─ gradio       : web UI served in Step 9
 
PACKAGES = [
    "peft>=0.10.0",
    "bitsandbytes>=0.43.0",
    "accelerate>=0.28.0",
    "transformers>=4.46.0",   # ≥4.46 for processing_class support
    "datasets>=2.19.0",
    "pypdf",
    "scikit-learn",
    "matplotlib",
    "gradio",
]
 
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir"] + PACKAGES
)
print("✓ All packages installed")
 
 
# ══════════════════════════════════════════════════════════════════════════════
# STEP 2 · Load Training Data  (PDFs → raw text corpus)
# ══════════════════════════════════════════════════════════════════════════════
 
import re
from pathlib import Path
from pypdf import PdfReader
 
# ── Paths ──────────────────────────────────────────────────────────────────────
PDF_DIR  = Path("/kaggle/input/datasets/sanjeevkv007/genainotes")
WORK_DIR = Path("/kaggle/working/notegenius_mistral")
WORK_DIR.mkdir(parents=True, exist_ok=True)
 
if not PDF_DIR.exists():
    raise FileNotFoundError(f"Dataset folder not found: {PDF_DIR}")
 
pdfs = sorted(PDF_DIR.rglob("*.pdf"))
if not pdfs:
    raise FileNotFoundError(f"No PDFs found in: {PDF_DIR}")
 
print(f"Found {len(pdfs)} PDF(s):")
for p in pdfs:
    print(f"  • {p.name}")
 
 
# ── Text cleaning ──────────────────────────────────────────────────────────────
 
def clean_text(x: str) -> str:
    """
    Remove null bytes, BOM markers, Unicode surrogates, and collapse whitespace.
 
    PDFs from math/science courses often embed surrogate code points (U+D800–U+DFFF)
    via symbol fonts.  encode→decode with errors='ignore' strips them silently
    before they can cause UnicodeEncodeError downstream.
    """
    if not x:
        return ""
    x = str(x).replace("\x00", " ").replace("\ufeff", " ")
    x = x.encode("utf-8", errors="ignore").decode("utf-8", errors="ignore")
    return re.sub(r"\s+", " ", x).strip()
 
 
# ── Extract all text from every PDF page ──────────────────────────────────────
# Unsupervised: raw note text is fed directly.  No labels needed.
# Mistral learns next-token prediction over your domain vocabulary.
 
raw_pages: list[str] = []
 
print("\nExtracting text from PDFs …")
for pdf_path in pdfs:
    reader = PdfReader(str(pdf_path))
    for page_idx, page in enumerate(reader.pages):
        try:
            text = page.extract_text() or ""
        except Exception:
            text = ""
        text = clean_text(text)
        if len(text) >= 50:
            raw_pages.append(text)
 
full_corpus = "\n\n".join(raw_pages)
 
print(f"✓ {len(raw_pages)} non-empty pages extracted")
print(f"✓ Total corpus size : {len(full_corpus):,} characters")
print(f"\nCorpus preview (first 300 chars):")
print(full_corpus[:300], "…")
 
# Save corpus — errors="ignore" drops any surrogates that slipped through
corpus_path = WORK_DIR / "corpus.txt"
corpus_path.write_text(full_corpus, encoding="utf-8", errors="ignore")
print(f"\n✓ Raw corpus saved → {corpus_path}")
 
 
# ══════════════════════════════════════════════════════════════════════════════
# STEP 3 · Preprocess Data  (tokenise into fixed-length windows)
# ══════════════════════════════════════════════════════════════════════════════
 
from transformers import AutoTokenizer
from datasets import Dataset
 
# ── Model ID ───────────────────────────────────────────────────────────────────
MODEL_ID   = "mistralai/Mistral-7B-Instruct-v0.2"
BLOCK_SIZE = 512   # token window per training example
                   # Mistral max is 32 768; 512 keeps GPU memory sane on T4
                   # and ensures many diverse training examples from the corpus
 
print(f"\nLoading Mistral tokeniser …")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
 
# Mistral has no dedicated pad token — reuse EOS (same pattern as GPT-2)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = "right"   # right-pad for causal LM training
 
 
# ── Tokenise the full corpus ───────────────────────────────────────────────────
print("Tokenising corpus …")
all_token_ids = tokenizer(
    full_corpus,
    add_special_tokens = False,
    return_tensors     = None,
)["input_ids"]
 
print(f"✓ Total tokens in corpus : {len(all_token_ids):,}")
 
 
# ── Slide a fixed window to create training blocks ────────────────────────────
# input_ids == labels — standard causal LM objective (predict next token).
 
def make_blocks(token_ids: list[int], block_size: int) -> list[dict]:
    """
    Chunk *token_ids* into non-overlapping windows of *block_size*.
    Each window is one training example: input_ids = labels = the block.
    """
    blocks = []
    for start in range(0, len(token_ids) - block_size, block_size):
        chunk = token_ids[start : start + block_size]
        blocks.append({
            "input_ids"     : chunk,
            "attention_mask": [1] * len(chunk),
            "labels"        : chunk,
        })
    return blocks
 
 
blocks = make_blocks(all_token_ids, BLOCK_SIZE)
print(f"✓ {len(blocks)} training blocks  (block size = {BLOCK_SIZE} tokens)")
 
full_dataset = Dataset.from_list(blocks)
 
# 90 / 10 train–eval split — reproducible via seed
split      = full_dataset.train_test_split(test_size=0.10, seed=42)
train_data = split["train"]
eval_data  = split["test"]
 
print(f"✓ Training blocks   : {len(train_data)}")
print(f"✓ Evaluation blocks : {len(eval_data)}")
 
 
# ══════════════════════════════════════════════════════════════════════════════
# STEP 4 · Set Up Training Arguments
# ══════════════════════════════════════════════════════════════════════════════
 
import torch
from transformers import TrainingArguments
 
OUTPUT_DIR = str(WORK_DIR / "checkpoints")
USE_GPU    = torch.cuda.is_available()
 
training_args = TrainingArguments(
    # ── Output ────────────────────────────────────────────────────────────────
    output_dir                   = OUTPUT_DIR,
 
    # ── Epochs ────────────────────────────────────────────────────────────────
    num_train_epochs             = 1,       # 3 epochs is a solid default for
                                            # unsupervised domain adaptation
 
    # ── Batch size & gradient accumulation ────────────────────────────────────
    # Mistral-7B in 4-bit NF4 uses ~8 GB VRAM per T4; batch=2 is safe.
    # Gradient accumulation gives an effective batch of 2×4=8 without OOM.
    per_device_train_batch_size  = 2,
    per_device_eval_batch_size   = 2,
    gradient_accumulation_steps  = 4,
    gradient_checkpointing       = True,    # trades compute for VRAM — essential on T4
 
    # ── Learning rate ─────────────────────────────────────────────────────────
    learning_rate                = 2e-4,    # standard LoRA recipe for 7B models
    lr_scheduler_type            = "cosine",
    warmup_ratio                 = 0.05,
 
    # ── Precision ─────────────────────────────────────────────────────────────
    fp16                         = USE_GPU, # mixed-precision on GPU
    use_cpu                      = not USE_GPU,  # graceful CPU fallback
 
    # ── Logging & evaluation ──────────────────────────────────────────────────
    logging_steps                = 10,
    eval_strategy                = "epoch",
    save_strategy                = "epoch",
    load_best_model_at_end       = True,
    metric_for_best_model        = "eval_loss",
 
    # ── Misc ──────────────────────────────────────────────────────────────────
    report_to                    = "none",
    dataloader_num_workers       = 2,
    seed                         = 42,
)
 
device_label = "T4 GPU (fp16 + 4-bit QLoRA)" if USE_GPU else "CPU (fp32)"
print(f"✓ Training arguments configured")
print(f"  Device              : {device_label}")
print(f"  Epochs              : {training_args.num_train_epochs}")
print(f"  Per-device batch    : {training_args.per_device_train_batch_size}")
print(f"  Gradient accum.     : {training_args.gradient_accumulation_steps}")
print(f"  Effective batch     : {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"  Learning rate       : {training_args.learning_rate}")
print(f"  Output dir          : {OUTPUT_DIR}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 222.8 MB/s eta 0:00:00
✓ All packages installed
Found 44 PDF(s):
  • BOOK-Generative-AI-CPP-Spuler-2024.pdf
  • CUDA-CPP-Debugginb-Book-Spuler-2024.pdf
  • CUDA-CPP-Optimization-Book-Spuler-2024.pdf
  • DS_merged_notes.pdf
  • Data Science.pdf
  • GenAI_Latest_notes.pdf
  • Generative-AI-Applications-Spuler-Sharpe-2024.pdf
  • Interview preparation - Day10.pdf
  • Interview preparation - Day11.pdf
  • Interview preparation - Day12.pdf
  • Interview preparation - Day13.pdf
  • Interview preparation - Day14.pdf
  • Interview preparation - Day15.pdf
  • Interview preparation - Day16.pdf
  • Interview preparation - Day17.pdf
  • Interview preparation - Day6.pdf
  • Interview preparation - Day7.pdf
  • Interview preparation - Day8.pdf
  • Interview preparation - Day9.pdf
  • Machine Learning with python.pdf
  • Python 3 For Absolute Beginners.pdf
  • Python Textbook.pdf
  • RAG-Optimization-Spuler-Sharpe-2025.pdf
  • SQL.pdf
  • Stati

incorrect startxref pointer(1)
parsing for Object Streams


✓ 9653 non-empty pages extracted
✓ Total corpus size : 17,951,971 characters

Corpus preview (first 300 chars):
Generative AI in C++ Coding Transformers and LLMs by David Spuler Yoryck AI

David Spuler ii Copyright © David Spuler, 2024. All rights reserved. Published by Yoryck AI Pty Ltd, Adelaide, Australia. https://www.yoryck.com ISBN: 979-8-871-92868-4 (Paperback) Printing: Version 1.0a, March 2024. First  …

✓ Raw corpus saved → /kaggle/working/notegenius_mistral/corpus.txt



Loading Mistral tokeniser …


config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

...

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

Tokenising corpus …
✓ Total tokens in corpus : 4,935,804
✓ 9640 training blocks  (block size = 512 tokens)
✓ Training blocks   : 8676
✓ Evaluation blocks : 964


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


✓ Training arguments configured
  Device              : T4 GPU (fp16 + 4-bit QLoRA)
  Epochs              : 1
  Per-device batch    : 2
  Gradient accum.     : 4
  Effective batch     : 8
  Learning rate       : 0.0002
  Output dir          : /kaggle/working/notegenius_mistral/checkpoints


In [2]:

# ══════════════════════════════════════════════════════════════════════════════
# STEP 5 · Initialise Model  (Mistral-7B 4-bit QLoRA + LoRA adapter)
# ══════════════════════════════════════════════════════════════════════════════
 
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType
 
# ── 4-bit NF4 quantisation config ─────────────────────────────────────────────
# NF4 (Normal Float 4) is the QLoRA paper's recommended quantisation type.
# Double quantisation further compresses the quantisation constants themselves,
# saving ~0.4 bits/param with negligible quality loss.
# Without this, Mistral-7B (~14 GB fp16) would not fit on a T4 (16 GB).
 
bnb_config = BitsAndBytesConfig(
    load_in_4bit              = True,
    bnb_4bit_quant_type       = "nf4",
    bnb_4bit_compute_dtype    = torch.float16,
    bnb_4bit_use_double_quant = True,
)
 
print("\nLoading Mistral-7B in 4-bit NF4 …  (this takes ~2 min)")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config = bnb_config,
    device_map          = "auto",       # Accelerate splits across T4 x2 automatically
    trust_remote_code   = True,
)
 
# Required before injecting LoRA: casts LayerNorm weights to fp32 for stability
base_model = prepare_model_for_kbit_training(base_model)
 
print(f"  Parameters (total)  : {sum(p.numel() for p in base_model.parameters()):,}")
 
 
# ── LoRA adapter configuration ────────────────────────────────────────────────
# Targeting Q, K, V, O projections gives richer gradient signal than Q+V alone.
# r=16 / alpha=32 (scaling=2) is the standard recipe for 7B models.
# lora_dropout=0.05 is light regularisation — appropriate for a raw-text corpus.
 
lora_config = LoraConfig(
    task_type       = TaskType.CAUSAL_LM,
    r               = 16,
    lora_alpha      = 32,
    target_modules  = ["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout    = 0.05,
    bias            = "none",
    inference_mode  = False,
)
 
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()
# Expected: ~1–2% trainable — only the LoRA matrices are updated
 


Loading Mistral-7B in 4-bit NF4 …  (this takes ~2 min)


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

  Parameters (total)  : 3,752,071,168
trainable params: 13,631,488 || all params: 7,255,363,584 || trainable%: 0.1879


In [3]:

# ══════════════════════════════════════════════════════════════════════════════
# STEP 6 · Train the Model
# ══════════════════════════════════════════════════════════════════════════════
 
from transformers import Trainer, DataCollatorForLanguageModeling
 
# mlm=False → causal LM (next-token prediction), not masked LM
data_collator = DataCollatorForLanguageModeling(
    tokenizer = tokenizer,
    mlm       = False,
)
 
print("\nStarting QLoRA fine-tuning …")
trainer = Trainer(
    model            = model,
    processing_class = tokenizer,   # replaces deprecated tokenizer= arg (≥4.46)
    args             = training_args,
    train_dataset    = train_data,
    eval_dataset     = eval_data,
    data_collator    = data_collator,
)
 
train_result = trainer.train()
 
print("\n✓ Training complete")
print(f"  Final training loss : {train_result.training_loss:.4f}")
print(f"  Total steps         : {train_result.global_step}")
print(f"  Runtime             : {train_result.metrics.get('train_runtime', 0):.0f}s")
 
 


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.



Starting QLoRA fine-tuning …


Epoch,Training Loss,Validation Loss
1,1.576835,1.695926



✓ Training complete
  Final training loss : 1.7469
  Total steps         : 1085
  Runtime             : 20585s


In [4]:
 
# ══════════════════════════════════════════════════════════════════════════════
# STEP 7 · Evaluate per Epoch  (loss curve + perplexity table)
# ══════════════════════════════════════════════════════════════════════════════
 
import math
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
 
log_history = trainer.state.log_history
 
# Separate step-level train logs from epoch-end eval logs
train_steps = [e["step"]      for e in log_history if "loss"      in e and "eval_loss" not in e]
train_loss  = [e["loss"]      for e in log_history if "loss"      in e and "eval_loss" not in e]
eval_steps  = [e["step"]      for e in log_history if "eval_loss" in e]
eval_loss   = [e["eval_loss"] for e in log_history if "eval_loss" in e]
eval_epochs = [e["epoch"]     for e in log_history if "eval_loss" in e]
 
# ── Loss curve plot ────────────────────────────────────────────────────────────
DARK_BG = "#0d0f14"; SURFACE = "#1c2030"; BORDER = "#2e3450"
TEXT    = "#e8eaf0"; MUTED   = "#7c849e"
BLUE    = "#6c8fff"; GREEN   = "#34d399"
 
fig, ax = plt.subplots(figsize=(10, 5))
fig.patch.set_facecolor(DARK_BG)
ax.set_facecolor(SURFACE)
 
ax.plot(train_steps, train_loss,
        color=BLUE, linewidth=1.5, label="Train loss", alpha=0.9)
ax.scatter(eval_steps, eval_loss,
           color=GREEN, s=70, zorder=5, label="Eval loss (epoch end)")
 
ax.yaxis.grid(True, color=BORDER, linewidth=0.6, linestyle="--", zorder=0)
ax.set_axisbelow(True)
ax.spines[:].set_visible(False)
ax.tick_params(colors=TEXT, labelsize=8.5)
ax.set_xlabel("Step",  color=TEXT, fontsize=9)
ax.set_ylabel("Loss",  color=TEXT, fontsize=9)
ax.set_title("Mistral-7B QLoRA fine-tuning loss  —  NoteGenius",
             color=TEXT, fontsize=11, fontweight="600", pad=12, loc="left")
ax.legend(frameon=True, facecolor=SURFACE, edgecolor=BORDER,
          labelcolor=TEXT, fontsize=9)
plt.tight_layout(pad=1.4)
 
plot_path = WORK_DIR / "loss_curve.png"
plt.savefig(plot_path, dpi=150, bbox_inches="tight")
plt.close()
print(f"\n✓ Loss curve saved → {plot_path}")
 
# ── Per-epoch perplexity table ─────────────────────────────────────────────────
# Perplexity = e^loss  (HF Trainer uses natural-log cross-entropy)
# Lower perplexity = model is less surprised by the notes vocabulary.
 
print("\n── Per-epoch evaluation results ──────────────────────────────────────────")
print(f"{'Epoch':>6}  │  {'Eval loss':>10}  │  {'Perplexity':>12}")
print("─" * 40)
for epoch, loss in zip(eval_epochs, eval_loss):
    print(f"{epoch:>6}  │  {loss:>10.4f}  │  {round(math.exp(loss), 3):>12.3f}")
 
# ── Qualitative generation probe ──────────────────────────────────────────────
# Prime the model with the opening of the corpus and inspect the completion.
 
from transformers import pipeline as hf_pipeline
 
probe_pipe = hf_pipeline(
    "text-generation",
    model          = model,
    tokenizer      = tokenizer,
    max_new_tokens = 80,
    temperature    = 0.7,
    do_sample      = True,
    pad_token_id   = tokenizer.eos_token_id,
)
 
probe_prompt = full_corpus[:120].strip()
print(f"\n── Qualitative generation probe ──────────────────────────────────────────")
print(f"Prompt : {probe_prompt}")
print(f"Output : {probe_pipe(probe_prompt)[0]['generated_text']}")
 

NameError: name 'trainer' is not defined

In [ ]:

# ══════════════════════════════════════════════════════════════════════════════
# STEP 8 · Save the Fine-Tuned Model
# ══════════════════════════════════════════════════════════════════════════════
 
from peft import PeftModel
 
ADAPTER_DIR = str(WORK_DIR / "adapter")       # LoRA adapter only   (~50–100 MB)
MERGED_DIR  = str(WORK_DIR / "merged_model")  # full merged model   (~14 GB fp16)
 
# ── Save LoRA adapter (fast, small) ───────────────────────────────────────────
print(f"\nSaving LoRA adapter → {ADAPTER_DIR}")
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print("✓ Adapter saved")
 
# ── Merge adapter into base and save full weights ─────────────────────────────
# Loading in fp16 for merging produces a ~14 GB file — skip if disk is tight.
print(f"\nMerging adapter into Mistral-7B → {MERGED_DIR}")
print("  (loading base in fp16 for merging — ~2 min) …")
 
merged_base  = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, device_map="auto"
)
merged_model = PeftModel.from_pretrained(merged_base, ADAPTER_DIR)
merged_model = merged_model.merge_and_unload()   # fuses LoRA weights in-place
 
merged_model.save_pretrained(MERGED_DIR, safe_serialization=True)
tokenizer.save_pretrained(MERGED_DIR)
print("✓ Merged model saved")
 
# ── Summary ────────────────────────────────────────────────────────────────────
print("\n── Saved artefacts ───────────────────────────────────────────────────────")
for path in [ADAPTER_DIR, MERGED_DIR]:
    size_gb = sum(f.stat().st_size for f in Path(path).rglob("*") if f.is_file()) / 1e9
    print(f"  {path}  ({size_gb:.2f} GB)")
 
 

In [3]:

# ══════════════════════════════════════════════════════════════════════════════
# STEP 9 · Deploy  (Gradio RAG chat with the fine-tuned Mistral-7B)
# ══════════════════════════════════════════════════════════════════════════════
#
# Deployment combines:
#   • TF-IDF retriever  — finds the most relevant PDF chunks for a query
#   • Fine-tuned Mistral — generates an answer grounded in those chunks
#
# Mistral is instruction-tuned so we use the [INST] prompt template —
# unlike GPT-2 which needed a raw completion-style prompt.
 
import pickle
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from matplotlib.ticker import FuncFormatter
import gradio as gr
 
 
# ── Chunk text for the retriever ──────────────────────────────────────────────
 
def chunk_text(text: str, chunk_size: int = 1200, overlap: int = 200) -> list[str]:
    """Slide a window over *text* and return overlapping chunks for retrieval."""
    text = clean_text(text)
    if len(text) < 40:
        return []
    chunks, start = [], 0
    while start < len(text):
        end   = min(start + chunk_size, len(text))
        chunk = clean_text(text[start:end])
        if len(chunk) >= 50:
            chunks.append(chunk)
        if end >= len(text):
            break
        start = max(end - overlap, start + 1)
    return chunks
 
 
# ── Rebuild TF-IDF retrieval index ────────────────────────────────────────────
print("\nBuilding TF-IDF retrieval index …")
 
deploy_docs: list[dict] = []
for pdf_path in pdfs:
    reader = PdfReader(str(pdf_path))
    for page_idx, page in enumerate(reader.pages):
        try:
            page_text = page.extract_text() or ""
        except Exception:
            page_text = ""
        for chunk in chunk_text(clean_text(page_text)):
            deploy_docs.append({
                "text"  : chunk,
                "source": pdf_path.name,
                "page"  : page_idx + 1,
            })
 
deploy_texts = [d["text"] for d in deploy_docs]
retriever    = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), max_features=50_000)
doc_matrix   = retriever.fit_transform(deploy_texts)
 
print(f"✓ Retrieval index ready — {len(deploy_texts)} chunks, "
      f"{doc_matrix.shape[1]} features")
 
 
# ── Load fine-tuned Mistral-7B for inference ──────────────────────────────────
# Prefer the merged model (no PEFT overhead); fall back to adapter if missing.
 
deploy_path = MERGED_DIR if Path(MERGED_DIR).exists() else ADAPTER_DIR
print(f"\nLoading fine-tuned model from: {deploy_path}")
 
ft_tokenizer = AutoTokenizer.from_pretrained(deploy_path, trust_remote_code=True)
ft_tokenizer.pad_token    = ft_tokenizer.eos_token
ft_tokenizer.padding_side = "right"
 
if Path(MERGED_DIR).exists():
    ft_model = AutoModelForCausalLM.from_pretrained(
        deploy_path,
        quantization_config = bnb_config,   # re-quantise for inference memory savings
        device_map          = "auto",
    )
else:
    ft_base  = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, quantization_config=bnb_config, device_map="auto"
    )
    ft_model = PeftModel.from_pretrained(ft_base, deploy_path)
 
ft_generator = hf_pipeline(
    "text-generation",
    model              = ft_model,
    tokenizer          = ft_tokenizer,
    max_new_tokens     = 220,
    temperature        = 0.1,       # near-deterministic for factual Q&A
    top_p              = 0.95,
    repetition_penalty = 1.08,
    do_sample          = True,
    pad_token_id       = ft_tokenizer.eos_token_id,
    return_full_text   = False,
)
print("✓ Fine-tuned Mistral-7B ready for inference")
 
 
# ── RAG helpers ───────────────────────────────────────────────────────────────
 
SYSTEM_PROMPT = (
    "You are NoteGenius, an expert class-notes assistant. "
    "Answer questions using only the provided context. "
    "If the answer is not in the context, say so honestly."
)
 
 
def retrieve_docs(query: str, top_k: int = 3) -> list[dict]:
    """Return the *top_k* most relevant chunks via TF-IDF cosine similarity."""
    q_vec = retriever.transform([clean_text(query)])
    sims  = cosine_similarity(q_vec, doc_matrix).flatten()
    return [
        {**deploy_docs[i], "score": float(sims[i])}
        for i in np.argsort(sims)[::-1][:top_k]
    ]
 
 
def answer_style_text(answer_length: str) -> str:
    return {
        "short": "Answer briefly in 2–4 lines.",
        "long" : "Answer in detail with multiple clear paragraphs.",
    }.get(answer_length.strip().lower(),
          "Answer in about one concise paragraph.")
 
 
def build_mistral_prompt(question: str, answer_length: str,
                          retrieved: list[dict], history: list[dict]) -> str:
    """
    Build a grounded Mistral [INST] prompt.
 
    Mistral is instruction-tuned, so using [INST]…[/INST] is essential —
    it's the format the model was trained to respond to.
    """
    context = "\n\n".join(
        f"[{d['source']} | Page {d['page']}]\n{d['text']}" for d in retrieved
    )
    history_text = "\n".join(
        f"{m['role'].capitalize()}: {m['content']}"
        for m in (history[-4:] or [])
    ) or "No previous conversation."
 
    user_turn = (
        f"{SYSTEM_PROMPT}\n\n"
        f"{answer_style_text(answer_length)}\n\n"
        f"Recent conversation:\n{history_text}\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {question}"
    )
    return f"<s>[INST] {user_turn} [/INST]"
 
 
def count_tokens(text: str) -> int:
    return len(ft_tokenizer.encode(text or "", add_special_tokens=False))
 
 
# ── Cost comparison data ───────────────────────────────────────────────────────
 
OPENAI_PRICING = {
    "GPT-5.4"     : {"input_per_m": 2.50, "output_per_m": 15.00},
    "GPT-4.1"     : {"input_per_m": 2.00, "output_per_m":  8.00},
    "GPT-4o mini" : {"input_per_m": 0.15, "output_per_m":  0.60},
}
 
TRADEOFF_DF = pd.DataFrame([
    {
        "Approach"          : "Fine-tuned Mistral-7B (this app)",
        "Cost pattern"      : "GPU / infra driven",
        "Quality"           : "Domain-adapted; strong reasoning",
        "Control & privacy" : "Full",
        "Ops burden"        : "Medium",
        "Best fit"          : "Private notes, rich answers, open-source",
    },
    {
        "Approach"          : "Vanilla Mistral-7B (no fine-tuning)",
        "Cost pattern"      : "GPU / infra driven",
        "Quality"           : "Good with RAG; no domain adaptation",
        "Control & privacy" : "Full",
        "Ops burden"        : "Low",
        "Best fit"          : "Quick start, general knowledge queries",
    },
    {
        "Approach"          : "OpenAI API",
        "Cost pattern"      : "Per-token billing",
        "Quality"           : "State-of-the-art; no fine-tuning",
        "Control & privacy" : "Lower",
        "Ops burden"        : "Lowest",
        "Best fit"          : "Fast launch, low maintenance",
    },
])
 
EMPTY_COST_DF = pd.DataFrame(columns=[
    "Model", "Type", "Input Tokens", "Output Tokens",
    "Input $/1M", "Output $/1M", "Estimated Cost ($)",
])
 
 
def build_cost_table(input_tokens, output_tokens,
                     gpu_hourly_cost, open_source_tokens_per_sec) -> pd.DataFrame:
    rows = [
        {
            "Model"              : m,
            "Type"               : "Paid API",
            "Input Tokens"       : input_tokens,
            "Output Tokens"      : output_tokens,
            "Input $/1M"         : p["input_per_m"],
            "Output $/1M"        : p["output_per_m"],
            "Estimated Cost ($)" : round(
                (input_tokens  / 1_000_000) * p["input_per_m"] +
                (output_tokens / 1_000_000) * p["output_per_m"], 6
            ),
        }
        for m, p in OPENAI_PRICING.items()
    ]
    secs = (input_tokens + output_tokens) / max(open_source_tokens_per_sec, 1e-6)
    rows.append({
        "Model"              : "Fine-tuned Mistral-7B (this app)",
        "Type"               : "Self-hosted",
        "Input Tokens"       : input_tokens,
        "Output Tokens"      : output_tokens,
        "Input $/1M"         : None,
        "Output $/1M"        : None,
        "Estimated Cost ($)" : round((secs / 3600.0) * gpu_hourly_cost, 6),
    })
    return pd.DataFrame(rows)
 
 
def build_cost_plot(cost_df: pd.DataFrame):
    """Dark-themed bar chart with per-bar dollar labels."""
    DARK_BG = "#0d0f14"; SURFACE = "#1c2030"; BORDER = "#2e3450"
    TEXT    = "#e8eaf0"; MUTED   = "#7c849e"
    BLUE    = "#6c8fff"; GREEN   = "#34d399"
 
    models = cost_df["Model"].astype(str).tolist()
    costs  = cost_df["Estimated Cost ($)"].astype(float).tolist()
    types  = cost_df["Type"].astype(str).tolist()
    colors = [GREEN if "Self" in t else BLUE for t in types]
    max_c  = max(costs) if costs else 1e-9
 
    fig, ax = plt.subplots(figsize=(9, 4.5))
    fig.patch.set_facecolor(DARK_BG); ax.set_facecolor(SURFACE)
 
    bars = ax.bar(range(len(models)), costs, color=colors,
                  width=0.55, edgecolor=BORDER, linewidth=0.8, zorder=3)
 
    for bar, cost in zip(bars, costs):
        label = f"${cost:.6f}" if cost < 0.01 else f"${cost:.5f}"
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + max_c * 0.03, label,
                ha="center", va="bottom", color=TEXT,
                fontsize=8.5, fontfamily="monospace", fontweight="bold")
 
    ax.yaxis.grid(True, color=BORDER, linewidth=0.6, linestyle="--", zorder=0)
    ax.set_axisbelow(True); ax.spines[:].set_visible(False)
    ax.set_xticks(list(range(len(models))))
    ax.set_xticklabels(models, color=TEXT, fontsize=8.5)
    ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"${v:.5f}"))
    ax.tick_params(colors=MUTED, labelsize=7.5)
    ax.set_ylim(0, max_c * 1.38)
    ax.set_title("estimated cost per request (USD)",
                 color=TEXT, fontsize=11, fontweight="600", pad=14, loc="left")
    ax.legend(handles=[mpatches.Patch(color=BLUE,  label="Paid API"),
                        mpatches.Patch(color=GREEN, label="Self-hosted")],
              frameon=True, facecolor=SURFACE, edgecolor=BORDER,
              labelcolor=MUTED, fontsize=8, loc="upper right")
    plt.tight_layout(pad=1.4)
    return fig
 
 
def build_summary_markdown(input_tokens, output_tokens, cost_df,
                            gpu_hourly_cost, open_source_tokens_per_sec) -> str:
    cheapest  = cost_df.sort_values("Estimated Cost ($)").iloc[0]
    costliest = cost_df.sort_values("Estimated Cost ($)", ascending=False).iloc[0]
    return f"""### Dynamic cost summary
- **Input tokens:** {input_tokens}
- **Output tokens:** {output_tokens}
- **Cheapest for this request:** {cheapest['Model']} — **${cheapest['Estimated Cost ($)']:.6f}**
- **Most expensive:** {costliest['Model']} — **${costliest['Estimated Cost ($)']:.6f}**
 
### Assumptions
- OpenAI costs use the pricing table defined in this notebook.
- Self-hosted cost: GPU @ **${gpu_hourly_cost}/hr**, throughput **{open_source_tokens_per_sec} tok/s**.
- This app uses a **fine-tuned Mistral-7B** — domain-adapted to your notes.""".strip()
 
 
# ── Main answer function (called by Gradio on every message) ──────────────────
 
def ask_notes(question: str, answer_length: str,
              gpu_hourly_cost: float, open_source_tokens_per_sec: float,
              history: list[dict]):
    history = history or []
    if not (question or "").strip():
        return history, "", EMPTY_COST_DF, TRADEOFF_DF, None, \
               "Ask a question to see the cost breakdown."
    try:
        retrieved  = retrieve_docs(question, top_k=3)
        prompt     = build_mistral_prompt(question, answer_length, retrieved, history)
        raw_answer = ft_generator(prompt)[0]["generated_text"].strip()
 
        seen, source_lines = set(), []
        for d in retrieved[:3]:
            key = (d["source"], d["page"])
            if key not in seen:
                seen.add(key)
                source_lines.append(f"• {d['source']} — Page {d['page']}")
 
        final_answer = raw_answer
        if source_lines:
            final_answer += "\n\nSources:\n" + "\n".join(source_lines)
 
        input_tokens  = count_tokens(prompt)
        output_tokens = count_tokens(raw_answer)
 
        cost_df    = build_cost_table(input_tokens, output_tokens,
                                      float(gpu_hourly_cost),
                                      float(open_source_tokens_per_sec))
        cost_plot  = build_cost_plot(cost_df)
        summary_md = build_summary_markdown(input_tokens, output_tokens, cost_df,
                                            float(gpu_hourly_cost),
                                            float(open_source_tokens_per_sec))
 
        history = history + [
            {"role": "user",      "content": f"{question}\n\n[Answer Type: {answer_length}]"},
            {"role": "assistant", "content": final_answer},
        ]
        return history, "", cost_df, TRADEOFF_DF, cost_plot, summary_md
 
    except Exception as e:
        history = history + [
            {"role": "user",      "content": question},
            {"role": "assistant", "content": f"⚠️ Error: {e}"},
        ]
        return history, "", EMPTY_COST_DF, TRADEOFF_DF, None, f"Error: {e}"
 
 
# ── Gradio UI (identical design to NoteGenius, updated branding) ───────────────
 
CSS = """
@import url('https://fonts.googleapis.com/css2?family=Playfair+Display:wght@500;700&family=DM+Sans:wght@300;400;500&family=DM+Mono:wght@400;500&display=swap');
:root{--bg:#0d0f14;--surface:#141720;--surface2:#1c2030;--surface3:#242840;
    --border:#2e3450;--accent:#6c8fff;--accent2:#a78bfa;--accent3:#34d399;
    --text:#e8eaf0;--muted:#7c849e;--gold:#f0c060;--user:#1e2d5a;--bot:#1a1f2e;
    --r:14px;--sh:0 8px 40px rgba(0,0,0,.5);}
body,.gradio-container{background:var(--bg)!important;font-family:'DM Sans',sans-serif!important;color:var(--text)!important;}
.gradio-container{background:radial-gradient(ellipse 80% 50% at 20% 10%,rgba(108,143,255,.07) 0%,transparent 60%),radial-gradient(ellipse 60% 40% at 80% 90%,rgba(167,139,250,.06) 0%,transparent 55%),var(--bg)!important;}
.hero{text-align:center;padding:40px 24px 20px;border-bottom:1px solid var(--border);margin-bottom:24px;}
.hero::after{content:'';display:block;width:120px;height:2px;background:linear-gradient(90deg,transparent,var(--accent),var(--accent2),transparent);margin:16px auto 0;}
.hero-title{font-family:'Playfair Display',serif!important;font-size:clamp(2rem,4vw,3rem)!important;font-weight:700!important;background:linear-gradient(135deg,#fff 30%,var(--accent) 70%,var(--accent2) 100%);-webkit-background-clip:text!important;-webkit-text-fill-color:transparent!important;background-clip:text!important;margin:0!important;}
.hero-sub{font-size:.9rem!important;color:var(--muted)!important;margin-top:8px!important;}
.pill{display:inline-flex;align-items:center;gap:6px;font-family:'DM Mono',monospace;font-size:.7rem;letter-spacing:1.5px;text-transform:uppercase;color:var(--accent);background:rgba(108,143,255,.1);border:1px solid rgba(108,143,255,.25);border-radius:20px;padding:4px 12px;margin-bottom:12px;}
details{background:var(--surface)!important;border:1px solid var(--border)!important;border-radius:var(--r)!important;overflow:hidden!important;box-shadow:var(--sh)!important;}
details>summary{background:var(--surface2)!important;padding:14px 20px!important;font-family:'DM Sans',sans-serif!important;font-size:.88rem!important;font-weight:500!important;color:var(--text)!important;border-bottom:1px solid var(--border)!important;cursor:pointer;list-style:none;transition:background .2s;}
details>summary:hover{background:var(--surface3)!important;}
textarea,input[type=text],input[type=number],select{background:var(--surface2)!important;border:1px solid var(--border)!important;border-radius:10px!important;color:var(--text)!important;font-family:'DM Sans',sans-serif!important;font-size:.9rem!important;padding:10px 14px!important;transition:border-color .2s,box-shadow .2s!important;}
textarea:focus,input:focus{border-color:var(--accent)!important;box-shadow:0 0 0 3px rgba(108,143,255,.15)!important;outline:none!important;}
ul[role=listbox],[data-testid=dropdown-options]{background:var(--surface2)!important;border:1px solid var(--border)!important;border-radius:10px!important;box-shadow:0 8px 32px rgba(0,0,0,.6)!important;overflow:hidden!important;padding:4px!important;}
li[role=option]{background:transparent!important;color:var(--text)!important;font-family:'DM Sans',sans-serif!important;font-size:.88rem!important;padding:9px 14px!important;border-radius:7px!important;cursor:pointer;}
li[role=option]:hover,li[role=option][aria-selected=true]{background:rgba(108,143,255,.15)!important;color:var(--accent)!important;}
label span{font-family:'DM Sans',sans-serif!important;font-size:.75rem!important;font-weight:500!important;color:var(--muted)!important;text-transform:uppercase!important;}
button.primary{background:linear-gradient(135deg,var(--accent),var(--accent2))!important;border:none!important;border-radius:10px!important;color:#fff!important;font-family:'DM Sans',sans-serif!important;font-weight:500!important;padding:10px 24px!important;cursor:pointer!important;box-shadow:0 4px 16px rgba(108,143,255,.3)!important;transition:opacity .2s,transform .15s!important;}
button.primary:hover{opacity:.88!important;transform:translateY(-1px)!important;}
button.secondary{background:var(--surface2)!important;border:1px solid var(--border)!important;border-radius:10px!important;color:var(--muted)!important;font-family:'DM Sans',sans-serif!important;transition:border-color .2s,color .2s!important;}
button.secondary:hover{border-color:#ff6b6b!important;color:#ff6b6b!important;}
.message.user{background:var(--user)!important;border:1px solid rgba(108,143,255,.2)!important;border-radius:12px 12px 4px 12px!important;color:var(--text)!important;font-family:'DM Sans',sans-serif!important;font-size:.9rem!important;line-height:1.6!important;padding:12px 16px!important;max-width:78%!important;align-self:flex-end!important;}
.message.bot{background:var(--bot)!important;border:1px solid var(--border)!important;border-radius:12px 12px 12px 4px!important;color:var(--text)!important;font-family:'DM Sans',sans-serif!important;font-size:.9rem!important;line-height:1.7!important;padding:14px 18px!important;max-width:82%!important;}
table{background:var(--surface)!important;border-collapse:collapse!important;font-family:'DM Mono',monospace!important;font-size:.8rem!important;width:100%!important;}
thead th{background:var(--surface3)!important;color:var(--accent)!important;font-size:.7rem!important;letter-spacing:1px!important;text-transform:uppercase!important;padding:11px 15px!important;border-bottom:2px solid var(--border)!important;white-space:nowrap!important;}
tbody tr{border-bottom:1px solid rgba(46,52,80,.6)!important;transition:background .15s;}
tbody tr:hover{background:rgba(108,143,255,.05)!important;}
tbody tr:nth-child(even){background:rgba(255,255,255,.025)!important;}
tbody td{color:var(--text)!important;padding:10px 15px!important;vertical-align:top!important;}
tbody td:last-child{color:var(--gold)!important;font-weight:600!important;}
[data-testid=plot]{background:var(--surface)!important;border:1px solid var(--border)!important;border-radius:var(--r)!important;overflow:hidden!important;padding:4px!important;}
.gr-markdown p,.gr-markdown li{font-family:'DM Sans',sans-serif!important;font-size:.88rem!important;color:var(--muted)!important;line-height:1.7!important;}
.gr-markdown h3{font-family:'Playfair Display',serif!important;font-size:1.05rem!important;color:var(--text)!important;margin-top:18px!important;border-left:3px solid var(--accent)!important;padding-left:10px!important;}
.gr-markdown strong{color:var(--gold)!important;}
.input-card{background:var(--surface)!important;border:1px solid var(--border)!important;border-radius:var(--r)!important;padding:16px 20px!important;box-shadow:var(--sh)!important;}
::-webkit-scrollbar{width:6px;height:6px;}
::-webkit-scrollbar-track{background:var(--surface);}
::-webkit-scrollbar-thumb{background:var(--border);border-radius:99px;}
::-webkit-scrollbar-thumb:hover{background:var(--accent);}
"""
 
theme = gr.themes.Base(
    primary_hue=gr.themes.Color(
        c50="#eef2ff",c100="#e0e7ff",c200="#c7d2fe",c300="#a5b4fc",
        c400="#818cf8",c500="#6c8fff",c600="#4f6ef7",c700="#3b55e6",
        c800="#2d43cc",c900="#1e2fa3",c950="#111b6b",
    ),
    neutral_hue=gr.themes.Color(
        c50="#f8f9fc",c100="#eceef5",c200="#d5d9e8",c300="#adb5cc",
        c400="#7c849e",c500="#5a6280",c600="#3e4566",c700="#2e3450",
        c800="#1c2030",c900="#141720",c950="#0d0f14",
    ),
    font=[gr.themes.GoogleFont("DM Sans"), "sans-serif"],
    font_mono=[gr.themes.GoogleFont("DM Mono"), "monospace"],
).set(
    body_background_fill                 = "#0d0f14",
    body_text_color                      = "#e8eaf0",
    block_background_fill                = "#141720",
    block_border_color                   = "#2e3450",
    block_radius                         = "14px",
    input_background_fill                = "#1c2030",
    button_primary_background_fill       = "linear-gradient(135deg,#6c8fff,#a78bfa)",
    button_primary_background_fill_hover = "linear-gradient(135deg,#8aa4ff,#c4b5fd)",
    button_primary_text_color            = "#ffffff",
    button_secondary_background_fill     = "#1c2030",
    button_secondary_text_color          = "#7c849e",
)
 
with gr.Blocks(title="NoteGenius", theme=theme, css=CSS) as demo:
 
    gr.HTML("""
    <div class="hero">
        <h1 class="hero-title">🧠 NoteGenius</h1>
        <p class="hero-sub">
            Fine-tuned Mistral-7B · QLoRA · RAG · 100% open-source
        </p>
    </div>
    """)
 
    with gr.Accordion("⚖️  Cost Breakdown — Fine-tuned Mistral-7B vs OpenAI API", open=False):
        gr.HTML('<div class="pill">⚙️ your assumptions</div>')
        gr.Markdown(
            "Numbers update live after each question. "
            "Adjust GPU rate and throughput to match your actual setup."
        )
        with gr.Row():
            gpu_hourly_cost = gr.Number(
                value=1.50, label="GPU cost ($/hr)",
                precision=3, minimum=0.01, maximum=50.0,
            )
            open_source_tokens_per_sec = gr.Number(
                value=120.0, label="Self-hosted throughput (tokens/sec)",
                precision=2, minimum=1.0, maximum=5000.0,
            )
        gr.HTML('<div class="pill">📊 summary</div>')
        summary_md = gr.Markdown("Ask something to see the breakdown.")
 
        gr.HTML('<div class="pill">🔄 approach tradeoffs</div>')
        tradeoff_table = gr.Dataframe(
            value=TRADEOFF_DF, interactive=False, wrap=True,
            row_count=(3, "fixed"), col_count=(6, "fixed"),
        )
        gr.HTML('<div class="pill">💰 per-request cost table</div>')
        cost_table = gr.Dataframe(value=EMPTY_COST_DF, interactive=False, wrap=True)
 
        gr.HTML('<div class="pill">📈 cost chart</div>')
        cost_plot = gr.Plot(label="")
 
    gr.HTML('<div class="pill" style="margin:20px 0 8px;">💬 ask your notes</div>')
    chatbot = gr.Chatbot(
        type="messages",
        height=520,
        label="",
        show_copy_button=True,
        bubble_full_width=False,
        avatar_images=(
            None,
            "https://api.dicebear.com/7.x/bottts-neutral/svg?seed=mistral7bnotes&backgroundColor=6c8fff",
        ),
        placeholder=(
            "<div style='text-align:center;color:#7c849e;padding:40px 20px;"
            "font-family:DM Sans,sans-serif;'>"
            "<div style='font-size:2.5rem;margin-bottom:12px;'>🧠</div>"
            "<div style='font-size:1rem;font-weight:500;color:#adb5cc;'>"
            "NoteGenius (Mistral-7B) is ready</div>"
            "<div style='font-size:.82rem;margin-top:6px;'>"
            "Ask anything from your uploaded notes.</div></div>"
        ),
    )
 
    with gr.Group(elem_classes="input-card"):
        with gr.Row():
            msg = gr.Textbox(
                placeholder="e.g.  What is backpropagation?  ·  Explain attention in transformers …",
                lines=1, max_lines=4, scale=6,
                autofocus=True, show_label=False,
            )
            answer_length = gr.Dropdown(
                choices=["Short", "Medium", "Long"],
                value="Medium", label="Answer length",
                scale=1, min_width=140,
            )
        with gr.Row():
            send_btn  = gr.Button("✦  Ask",   variant="primary",   scale=3)
            clear_btn = gr.Button("✕  Clear", variant="secondary", scale=1)
 
    gr.HTML("""
    <div style="text-align:center;padding:24px 0 12px;color:#3e4566;
                font-family:'DM Mono',monospace;font-size:.68rem;letter-spacing:1px;">
        NoteGenius &nbsp;·&nbsp; Fine-tuned Mistral-7B + QLoRA + TF-IDF RAG &nbsp;·&nbsp; Kaggle
    </div>
    """)
 
    _in  = [msg, answer_length, gpu_hourly_cost, open_source_tokens_per_sec, chatbot]
    _out = [chatbot, msg, cost_table, tradeoff_table, cost_plot, summary_md]
 
    msg.submit(ask_notes, inputs=_in, outputs=_out)
    send_btn.click(ask_notes, inputs=_in, outputs=_out)
 
    def clear_all():
        return [], "", EMPTY_COST_DF, TRADEOFF_DF, None, "Ask something to see the breakdown."
 
    clear_btn.click(clear_all, inputs=[], outputs=_out)
 
 
# ── Launch ─────────────────────────────────────────────────────────────────────
demo.queue(default_concurrency_limit=1)
demo.launch(share=True, debug=False, show_error=True)
# Public URL is printed here — share it in your LinkedIn post!
 



Building TF-IDF retrieval index …


NameError: name 'pdfs' is not defined